# 🧠 Canonical LangGraph ReAct Agent with Cohzen (`cz`) Observability

This notebook demonstrates:
1. **Connecting to your custom LLM endpoint** using OpenAI client and LangChain `ChatOpenAI`.
2. **Building the canonical LangGraph ReAct Agent**: Dynamic tool calling (`@tool`), conditional routing (`tools_condition`), and checkpointer persistence (`MemorySaver`).
3. **Extensively testing on famous repositories** (e.g. `langchain-ai/langgraph`).
4. **Applying Cohzen (`cz`)**:
   - Static architecture reconstruction (`cz scan`)
   - Observability & telemetry governance audit (`cz audit`)
   - Zero-overhead runtime execution recording into `.cohzen/executions.db` (`cz runs`, `cz run <id>`)
   - Interactive Mermaid flowchart visualization

In [1]:
# Baseline test using raw OpenAI client (as originally highlighted)
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(
    base_url=os.getenv("API_BASE_URL"),
    api_key=os.getenv("API_KEY"),
)

stream = client.chat.completions.create(
    model=os.getenv("LLM_MODEL", "local-fast"),
    messages=[
        {
            "role": "user",
            "content": "Explain AI observability in detail."
        }
    ],
    max_tokens=200,
    stream=True,
    extra_body={
        "think": False
    },
)

for chunk in stream:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)
print()

**AI Observability** refers to the ability to monitor, analyze, and understand the behavior and performance of artificial intelligence (AI) systems, including machine learning models, deep learning networks, and other AI-driven applications. It is crucial for ensuring that AI systems are reliable, transparent, and performant, especially in real-world scenarios where unexpected behaviors or errors can occur.

---

### 🔍 **What is AI Observability?**

AI Observability is the process of gaining insights into the internal state and behavior of AI systems, similar to how traditional observability techniques (like monitoring, logging, tracing) are used in software systems. It involves:

- **Monitoring**: Tracking the performance and behavior of AI models and systems over time.
- **Logging**: Capturing events, inputs, outputs, and model states during inference or training.
- **Tracing**: Tracking the flow of data through the AI pipeline, including input, processing, and output.
- **Metrics**:

## 🔌 1. Connect LangChain & Enable Cohzen Runtime Telemetry

We configure `ChatOpenAI` pointing to the exact same local endpoint, passing `extra_body={"think": False}` to disable thinking mode, and enable `cz.execution.install()`.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

# Auto-install Cohzen runtime instrumentation
import cz
import cz.execution
cz.execution.install()

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url=os.getenv("API_BASE_URL"),
    api_key=os.getenv("API_KEY"),
    model=os.getenv("LLM_MODEL", "local-fast"),
    extra_body={"think": False},
    temperature=0.1,
)

test_res = llm.invoke("Say 'Cohzen + LangGraph ready!' in 4 words.")
print("LLM Connection status:", test_res.content)

LLM Connection status: Cohzen LangGraph ready!


## 🛠️ 2. Define Repository Intelligence Tools (`@tool` + `@cz.observe`)

We provide the agent with tools to inspect repository layouts, read source files, and search symbols. Decorating with `@cz.observe` gives 100% observability attribution in `cz audit`.

In [3]:
from pathlib import Path
from langchain_core.tools import tool

@tool
@cz.observe(
    name="list_repo_structure",
    kind="tool",
    entity_id="tool.repo.list_structure",
    reason=lambda target_dir=".": f"Inspecting repository directory: {target_dir}",
)
def list_repo_structure(target_dir: str = ".") -> str:
    """List directories and key files inside a repository path."""
    p = Path(target_dir).resolve()
    if not p.exists():
        return f"Error: Path '{target_dir}' does not exist."
    if not p.is_dir():
        return f"Error: Path '{target_dir}' is a file, not a directory."
    items = []
    try:
        for entry in sorted(p.iterdir()):
            if entry.name.startswith(".git"):
                continue
            kind = "DIR " if entry.is_dir() else "FILE"
            rel_path = f"{target_dir.rstrip('/')}/{entry.name}"
            items.append(f"{kind}  {rel_path}")
        return "\n".join(items[:40]) or "(Empty directory)"
    except Exception as e:
        return f"Error: {e}"

@tool
@cz.observe(
    name="read_file_snippet",
    kind="tool",
    entity_id="tool.repo.read_snippet",
    reason=lambda file_path, max_lines=40: f"Reading file snippet: {file_path}",
)
def read_file_snippet(file_path: str, max_lines: int = 40) -> str:
    """Read the first few lines of a file in the repository."""
    p = Path(file_path).resolve()
    if not p.is_file():
        alt = Path("benchmarks/famous_repos/langgraph") / file_path
        if alt.is_file():
            p = alt.resolve()
        else:
            return f"Error: File '{file_path}' does not exist."
    try:
        lines = []
        with open(p, "r", encoding="utf-8", errors="replace") as f:
            for i, line in enumerate(f):
                if i >= max_lines:
                    lines.append(f"... (truncated after {max_lines} lines)")
                    break
                lines.append(line.rstrip("\n"))
        return "\n".join(lines)
    except Exception as e:
        return f"Error: {e}"

REPO_TOOLS = [list_repo_structure, read_file_snippet]
print(f"Defined {len(REPO_TOOLS)} repository intelligence tools.")

Defined 2 repository intelligence tools.


## 🔄 3. Build Canonical LangGraph ReAct Graph

We assemble the canonical ReAct agent:
- `agent`: LLM call with bound tools
- `tools`: `ToolNode` executing tools
- `tools_condition`: Dynamic conditional routing
- `MemorySaver`: State checkpointing

In [4]:
from typing import Annotated, Any, Dict, Sequence, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

@cz.observe(
    name="agent",
    kind="node",
    entity_id="node.agent",
    reason="Evaluate repository state and decide next tool action or final answer.",
)
def call_agent(state: AgentState) -> Dict[str, Any]:
    model_with_tools = llm.bind_tools(REPO_TOOLS)
    cz.reason("Invoking LLM to analyze state and select next tool or synthesize response.")
    response = model_with_tools.invoke(state["messages"])
    return {"messages": [response]}

tool_node = ToolNode(REPO_TOOLS)

# Construct graph
workflow = StateGraph(AgentState)
workflow.add_node("agent", call_agent)
workflow.add_node("tools", tool_node)
workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", tools_condition, {"tools": "tools", END: END})
workflow.add_edge("tools", "agent")

memory = MemorySaver()
repo_agent = workflow.compile(checkpointer=memory)
repo_agent.name = "repo_react_agent"
print("✔ Successfully compiled repo_react_agent with MemorySaver.")

✔ Successfully compiled repo_react_agent with MemorySaver.


## 🚀 4. Execute the Agent against a Famous Repository

We invoke the agent live to inspect `benchmarks/famous_repos/langgraph`.

In [7]:
config = {"configurable": {"thread_id": "nb-session-1"}, "recursion_limit": 10}
system_prompt = (
    "You are an expert software architecture agent. You have access to tools to "
    "inspect repository directories and read file snippets.\n"
    "Inspect 'benchmarks/famous_repos/langgraph/README.md' and summarize what LangGraph is."
)

initial_state = {
    "messages": [
        SystemMessage(content=system_prompt),
        HumanMessage(content="Read benchmarks/famous_repos/langgraph/README.md and summarize what it does."),
    ]
}

result = repo_agent.invoke(initial_state, config=config)
print("\n--- Agent Final Answer ---")
print(result["messages"][-1].content)


--- Agent Final Answer ---
{"error": "The file path 'benchmarks/famous_repos/langgraph/README.md' does not exist in the current working directory. Please ensure that the file exists before running the `read_file_snippet` function.", "solution": "To use the `read_file_snippet` function, you need to have the file 'benchmarks/famous_repos/langgraph/README.md' available in your working directory. You can create it manually or use a tool like `nano` or `vim` to edit it. Once the file is present, you can run the function again."}


## 📊 5. Cohzen (`cz`) Observability & Architecture Audit

Now we apply `cz` to:
1. **Static scan** our agent's architecture.
2. **Audit** observability instrumentation.
3. **Query recorded executions** directly from `.cohzen/executions.db`.

In [8]:
from cz.scanner.engine import scan_repository
from cz.formatters.console import print_console_report, print_audit_report
from cz.formatters.mermaid import format_mermaid
from cz.execution.sqlite import SQLiteExecutionStore
from pathlib import Path

# 1. Static Scan of our new agent
scan_res = scan_repository("examples/react_repo_agent")
print_console_report(scan_res)

# 2. Observability Audit
print_audit_report(scan_res)

# 3. Mermaid Flow Diagram
print("\n--- Mermaid Flow Diagram ---")
print(format_mermaid(scan_res))

# 4. Query recorded runs from Cohzen SQLite DB
store = SQLiteExecutionStore(db_path=Path(".cohzen/executions.db"))
runs = store.list_executions(limit=5)
print(f"\nTotal recorded executions: {len(runs)}")
for r in runs:
    print(f"  • Run ID: {r.id[:8]} | Graph: {r.graph_name} | Status: {r.status} | Duration: {r.duration_ms:.1f}ms")

╭────────────────────────────────────── Cohzen Architecture Scanner (v0.1) ───────────────────────────────────────╮
│                                                                                                                 │
│  Scanned Target: /Users/aarora/Dev/cohzen-ai/cz/examples/react_repo_agent                                       │
│  Frameworks:     ✓ langgraph                                                                                    │
│  Files Examined: 5 total (2 Python files)                                                                       │
│  Scan Duration:  4.89 ms                                                                                        │
│                                                                                                                 │
│  Graphs Detected: 1  • Compiled: 1  • Registered Nodes: 2  • Edges/Routes: 3                                    │
│  Node Handlers:   1  • LLM Call Sites: 1  • Tools Defined: 3  • ToolNodes: 1                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────── Graph #1: workflow (StateGraph) StateSchema: AgentState ────────────────────────────╮
│ ✔ COMPILED -> variable: compiled_agent at app.py:192                                                            │
│   Checkpointer: memory                                                                                          │
╰────────────────────────────────────────────────── app.py:171 ───────────────────────────────────────────────────╯

╭────────────────────── 🌐 Architecture Flow (Nodes • Edges • Tools • Checkpointer • LLMs) ───────────────────────╮
│                                                                                                                 │
│    💾 Checkpointer: memory (State Persistence)                                                                  │
│          ┆                                                                                                      │
│          ▼                                                                                                      │
│     ● START                                                                                                     │
│       │                                                                                                         │
│       ▼                                                                                                         │
│       ├── ? route: 'tools' ──► tools                                                                            │
│       ├── ? route: 'END' ──►  ● END                                                                             │
│    ╔══ 🛠️ tools (ToolNode) ═══════════════════════════════╗                                                     │
│    ║  Tools: list_repo_structure, read_file_snippet                                                             │
│    ╚══════════════════════════════════════════════════════╝                                                     │
│    ┌── 🤖 tools (tool_node) ──────────────────────┐                                                             │
│    │  🧠 LLM: None (logic/router)                                                                               │
│    └───────────────────────────────────────────────────────┘                                                    │
│       └──► agent                                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                            Registered Nodes & Handlers                                            
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Node Name ┃ Handler / Location          ┃ LLM Calls & Models                ┃ Tools Bound / Attached            ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 🤖 agent  │ call_agent (app.py:149-162) │ langchain                         │ bound:                            │
│           │                             │ (model_with_tools.invoke)         │                                   │
│ 🤖 tools  │ tool_node (app.py:175)      │ Tool Execution                    │ ToolNode(list_repo_structure,     │
│           │                             │                                   │ read_file_snippet)                │
└───────────┴─────────────────────────────┴───────────────────────────────────┴───────────────────────────────────┘

                                       Graph Topology & Routing Edges                                       
┏━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Type        ┃ Source ┃ Direction ┃ Target / Routing                                         ┃ Location   ┃
┡━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ DIRECT      │ START  │    ──>    │ agent                                                    │ app.py:178 │
│ CONDITIONAL │ agent  │   ─?─>    │ fn: tools_condition {'tools' -> 'tools', 'END' -> 'END'} │ app.py:181 │
│ DIRECT      │ tools  │    ──>    │ agent                                                    │ app.py:188 │
└─────────────┴────────┴───────────┴──────────────────────────────────────────────────────────┴────────────┘

...

╭───────────────────────────────── Cohzen Observability & Telemetry Audit (v0.1) ─────────────────────────────────╮
│                                                                                                                 │
│  Scanned Target:    /Users/aarora/Dev/cohzen-ai/cz/examples/react_repo_agent                                    │
│  Files Examined:    5 total (2 Python files)                                                                    │
│  Scan Duration:     4.89 ms                                                                                     │
│                                                                                                                 │
│  Observability Tools:  ✔ Langfuse                                                                               │
│  Telemetry Packages:  None detected in package manifests                                                        │
│  Environment Config:  No tracing environment variables found in .env files                                      │
│  Tracing Coverage:    Nodes: 1/1 (100.0%)  •  Tools: 3/3                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                           Node Instrumentation Status                                            
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Node / Function ┃ Location       ┃ Instrumentation Status ┃ Observability Tool ┃ Active Decorators ┃ LLM Calls ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ 🤖 call_agent   │ app.py:149-162 │ ✔ Instrumented         │ Langfuse           │ observe           │ 1 call(s) │
└─────────────────┴────────────────┴────────────────────────┴────────────────────┴───────────────────┴───────────┘

                                 Tool Instrumentation Status                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Tool Name              ┃ Location      ┃ Tracing Status ┃ Observability Tool ┃ Decorators ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ 🔧 list_repo_structure │ app.py:43-63  │ ✔ Instrumented │ Langfuse           │ observe    │
│ 🔧 read_file_snippet   │ app.py:68-89  │ ✔ Instrumented │ Langfuse           │ observe    │
│ 🔧 search_repo_symbols │ app.py:99-121 │ ✔ Instrumented │ Langfuse           │ observe    │
└────────────────────────┴───────────────┴────────────────┴────────────────────┴────────────┘


--- Mermaid Flow Diagram ---
%% ========================================================
%% LangGraph Architecture: workflow (StateGraph)
%% State Schema: AgentState
%% Defined at: /Users/aarora/Dev/cohzen-ai/cz/examples/react_repo_agent/app.py:171
%% Compiled at line 192 -> compiled_agent (Checkpointer: memory)
%% ========================================================
flowchart TD
    __start__([● START]):::sentinelNode
    __end__([● END]):::sentinelNode
    checkpointer["💾 Checkpointer<br/><b>memory</b><br/><i>State Persistence</i>"]:::checkpointerNode
    checkpointer -.->|state persistence| __start__
    agent["🤖 <b>agent</b><br/>Handler: call_agent<br/>🧠 LLM: langchain (model_with_tools.invoke)<br/>🛠️ Tools: [list_repo_structure, read_file_snippet]<br/>🔭 Tracing: @observe"]:::instrumentedAgent
    tools[["🛠️ <b>tools</b> (ToolNode)<br/>Tools: list_repo_structure, read_file_snippet<br/>🔭 Tracing: 2/2 instrumented"]]:::toolNode
    __start__ --> agent
    agent -.->|route: tools

AttributeError: 'Execution' object has no attribute 'graph_name'